In [58]:
import osmium
from osmium import osm, filter
from osmium.filter import TagFilter
import h3
import json

In [59]:
from pathlib import Path

path = Path("data/osm/california-260203.osm.pbf/").resolve()

print(path)

C:\Users\ddavi\Projects\scenic-route\processing\src\data\osm\california-260203.osm.pbf


In [60]:
water_data = json.load(open("constants/water_tags.json"))
water_tags = [v for (k, v) in water_data]

vegetation_data = json.load(open("constants/vegetation_tags.json"))
vegetation_tags = [v for (k, v) in vegetation_data]

geological_data = json.load(open("constants/natural_geology_related_tags.json"))
geological_tags = [v for (k, v) in geological_data]

waterway_data = json.load(open("constants/waterway_tags.json"))
waterway_tags = [v for (k, v) in waterway_data]

print(f"Water tags: {water_tags}")
print(f"Vegetation tags: {vegetation_tags}")
print(f"Geological tags: {geological_tags}")
print(f"Waterway tags: {waterway_tags}")

Water tags: ['bay', 'beach', 'blowhole', 'cape', 'coastline', 'crevasse', 'geyser', 'glacier', 'hot_spring', 'isthmus', 'mud', 'peninsula', 'reef', 'shingle', 'shoal', 'spring', 'strait', 'water', 'wetland']
Vegetation tags: ['fell', 'grassland', 'heath', 'moor', 'scrub', 'shrubbery', 'tree', 'tree_row', 'tundra', 'wood']
Geological tags: ['arch', 'arete', 'bare_rock', 'blockfield', 'cave_entrance', 'cliff', 'dune', 'earth_bank', 'fumarole', 'hill', 'peak', 'ridge', 'rock', 'saddle', 'sand', 'scree', 'sinkhole', 'stone', 'valley', 'volcano']
Waterway tags: ['river', 'riverbank', 'stream', 'tidal_channel', 'canal', 'ditch', 'fairway', 'dam', 'weir', 'waterfall']


In [61]:
class ScenicHandler(osmium.SimpleHandler):
    def __init__(self, resolution=8):
        super().__init__()
        self.resolution = resolution
        self.cells = {}  # h3_cell_id -> dict of counts

    def _get_cell(self, lat, lng):
        # Get the H3 cell for the given lat/lng, creating an entry if it doesn't exist.
        cell = h3.latlng_to_cell(lat, lng, self.resolution)
        if cell not in self.cells:
            self.cells[cell] = {
                "water": 0,
                "landcover": 0,
                "relief": 0,
                "recreation": 0,
                "viewpoint": 0,
                "urban": 0,
            }
        return self.cells[cell]

    def node(self, n: osm.Node):
        if not n.location.valid():
            return
        lat, lng = n.location.lat, n.location.lon
        tags = n.tags

        if tags.get("tourism") == "viewpoint":
            self._get_cell(lat, lng)["viewpoint"] += 1
        elif tags.get("natural") == "peak":
            self._get_cell(lat, lng)["relief"] += 1

    def way(self, w: osm.Way):
        # Only handle linear water features here — forests/parks/urban are
        # closed ways that osmium also sends to area(), so we handle them
        # there to avoid double-counting.
        tags = w.tags
        if not (
            tags.get("natural") in water_tags
            or tags.get("waterway") in waterway_tags
        ):
            return

        nodes = [n for n in w.nodes if n.location.valid()]
        if not nodes:
            return
        lat = sum(n.location.lat for n in nodes) / len(nodes)
        lng = sum(n.location.lon for n in nodes) / len(nodes)
        self._get_cell(lat, lng)["water"] += 1


    def area(self, a: osm.Area):
        tags = a.tags

        if tags.get("landuse") == "forest" or tags.get("natural") in vegetation_tags:
            feature = "landcover"
        elif (
            tags.get("leisure") in ("park", "nature_reserve")
            or tags.get("boundary") == "protected_area"
        ):
            feature = "recreation"
        elif tags.get("landuse") in ("industrial", "commercial"):
            feature = "urban"
        elif tags.get("natural") in geological_tags:
            feature = "relief"
        else:
            return

        try:
            outer = next(a.outer_rings())
            nodes = [n for n in outer if n.location.valid()]
            if not nodes:
                return

            outer_coords = [(n.location.lat, n.location.lon) for n in nodes]
            holes = [
                [(n.location.lat, n.location.lon) for n in ring if n.location.valid()]
                for ring in a.inner_rings(outer)
            ]

            polygon = h3.LatLngPoly(outer_coords, *holes)
            touched = h3.polygon_to_cells(polygon, self.resolution)

            if not touched:
                # Area too small to fill any cell — fall back to centroid
                lat = sum(n.location.lat for n in nodes) / len(nodes)
                lng = sum(n.location.lon for n in nodes) / len(nodes)
                touched = [h3.latlng_to_cell(lat, lng, self.resolution)]

        except (StopIteration, AttributeError):
            return

        for cell in touched:
            if cell not in self.cells:
                self.cells[cell] = {
                    "water": 0,
                    "landcover": 0,
                    "relief": 0,
                    "recreation": 0,
                    "viewpoint": 0,
                    "urban": 0,
                }
            self.cells[cell][feature] += 1

### Execute
Runtime (idx, filters): 4m 27.3s  
Runtime (filters): 3m 53.8s  
Runtime (idx): forever  

In [ ]:
RESOLUTION = 8
handler = ScenicHandler(resolution=RESOLUTION)
handler.apply_file(
    path,
    locations=True,
    # idx="sparse_file_array",
    filters=[
        TagFilter(
            ("tourism", "viewpoint"),
            *[(("waterway", tag)) for tag in waterway_tags],
            *[(("natural", tag)) for tag in vegetation_tags],
            *[(("natural", tag)) for tag in geological_tags],
            ("landuse", "forest"),
            ("landuse", "industrial"),
            ("landuse", "commercial"),
            ("leisure", "park"),
            ("leisure", "nature_reserve"),
            ("boundary", "protected_area"),
        )
    ],
)

print(f"Parsed {len(handler.cells)} H3 cells")

Parsed 355664 H3 cells


### Runtime: 

In [63]:
import json

with open("data/output/scenic_cells.json", "w") as f:
    json.dump(handler.cells, f)

print(f"Saved {len(handler.cells)} H3 cells")

Saved 355664 H3 cells


In [64]:
import pandas as pd
import json
import os

print(os.getcwd())

c:\Users\ddavi\Projects\scenic-route\processing\src


In [65]:
# with open("data/output/scenic_cells_v1.json", "r") as f:
#     cells = json.load(f)

cells = handler.cells

In [71]:
# Convert to DataFrame for easy scoring
df = pd.DataFrame.from_dict(cells, orient="index")
df.index.name = "h3_cell"
df.reset_index(inplace=True)

# Scenic score formula
df["diversity"] = (df[["water", "landcover", "relief", "recreation", "viewpoint"]] > 0).sum(axis=1)
feature_cols = ["water", "landcover", "relief", "recreation", "viewpoint", "urban"]
df[feature_cols] = df[feature_cols].clip(upper=1)
df["raw_score"] = (
    3 * df["water"]
    + 2 * df["landcover"]
    + 3 * df["relief"]
    + 2 * df["recreation"]
    + 1 * df["viewpoint"]
    + 2 * df["diversity"]
    - 2 * df["urban"]
)


# Normalize to 0–100
df["score"] = (
    (df["raw_score"] - df["raw_score"].min())
    / (df["raw_score"].max() - df["raw_score"].min())
    * 100
).clip(0, 100)

df_ranked = df.sort_values("score", ascending=False)
print(df_ranked[["h3_cell", "score"]].head(200))

               h3_cell       score
5365   8828308d43fffff  100.000000
1843   88291249c3fffff  100.000000
5863   88283099c9fffff  100.000000
6635   8829ab6e45fffff  100.000000
41     8829a18aa5fffff  100.000000
...                ...         ...
64707  8829aa3753fffff   86.956522
64598  8829aa3601fffff   86.956522
5442   8828340b37fffff   86.956522
5444   882834ca23fffff   86.956522
5414   8828340f3bfffff   86.956522

[200 rows x 2 columns]


In [75]:
from datetime import datetime

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
df_ranked.to_json(f"data/output/scenic_scores_{ts}.json", orient="records")
df_ranked.to_csv(f"data/output/scenic_scores_{ts}.csv", index=False)

In [68]:
print(df_ranked.columns.tolist())

['h3_cell', 'water', 'landcover', 'relief', 'recreation', 'viewpoint', 'urban', 'diversity', 'raw_score', 'score']


In [73]:
# Detect skew
print(df["raw_score"].describe())
print()
print(df["raw_score"].quantile([0.5, 0.75, 0.9, 0.95, 0.99]))


count    355664.000000
mean          5.564626
std           2.554935
min          -2.000000
25%           4.000000
50%           5.000000
75%           5.000000
max          21.000000
Name: raw_score, dtype: float64

0.50     5.0
0.75     5.0
0.90     9.0
0.95     9.0
0.99    14.0
Name: raw_score, dtype: float64
